In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
data_dir = '/content/drive/MyDrive/nba-mvp-predictor/data'
os.makedirs(data_dir, exist_ok=True)
print(os.listdir(data_dir))

Mounted at /content/drive
['01_setup.ipynb']


In [2]:
!pip install kaggle xgboost streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 73.0 MB/s eta 0:00:00


In [9]:
from google.colab import files
uploaded = files.upload()  # select kaggle.json from your computer when prompted

Saving kaggle.json to kaggle.json


In [10]:
!ls -la /content

total 28
drwxr-xr-x 1 root root 4096 Sep 15 14:27 .
drwxr-xr-x 1 root root 4096 Sep 15 14:03 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:25 .config
drwx------ 5 root root 4096 Sep 15 14:07 drive
-rw-r--r-- 1 root root   69 Sep 15 14:27 kaggle.json
-rw-r--r-- 1 root root   37 Sep 15 14:14 Kaggle.txt
drwxr-xr-x 1 root root 4096 Sep  4 13:25 sample_data


In [11]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [12]:
!kaggle datasets download -d sumitrodatta/nba-aba-baa-stats -p /content/drive/MyDrive/nba-mvp-predictor/data --unzip

Dataset URL: https://www.kaggle.com/datasets/sumitrodatta/nba-aba-baa-stats
License(s): CC0-1.0
100% 10.6M/10.6M [00:00<00:00, 46.9MB/s]



In [13]:
import pandas as pd
import os

data_dir = '/content/drive/MyDrive/nba-mvp-predictor/data'
files = os.listdir(data_dir)
print(files)

['01_setup.ipynb', 'Advanced.csv', 'All-Star Selections.csv', 'Draft Pick History.csv', 'End of Season Teams (Voting).csv', 'End of Season Teams.csv', 'Opponent Stats Per 100 Poss.csv', 'Opponent Stats Per Game.csv', 'Opponent Totals.csv', 'Per 100 Poss.csv', 'Per 36 Minutes.csv', 'Player Award Shares.csv', 'Player Career Info.csv', 'Player Per Game.csv', 'Player Play By Play.csv', 'Player Season Info.csv', 'Player Shooting.csv', 'Player Totals.csv', 'Team Abbrev.csv', 'Team Stats Per 100 Poss.csv', 'Team Stats Per Game.csv', 'Team Summaries.csv', 'Team Totals.csv']


In [15]:
per_game = pd.read_csv(f'{data_dir}/Player Per Game.csv')
advanced = pd.read_csv(f'{data_dir}/Advanced.csv')
team_stats = pd.read_csv(f'{data_dir}/Team Stats Per Game.csv')
voting = pd.read_csv(f'{data_dir}/End of Season Teams (Voting).csv')

for name, df in [('per_game', per_game), ('advanced', advanced), ('team_stats', team_stats), ('voting', voting)]:
    print(name, df.shape)
    print(df.columns.tolist())
    print()

per_game (33339, 32)
['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs', 'mp_per_game', 'fg_per_game', 'fga_per_game', 'fg_percent', 'x3p_per_game', 'x3pa_per_game', 'x3p_percent', 'x2p_per_game', 'x2pa_per_game', 'x2p_percent', 'e_fg_percent', 'ft_per_game', 'fta_per_game', 'ft_percent', 'orb_per_game', 'drb_per_game', 'trb_per_game', 'ast_per_game', 'stl_per_game', 'blk_per_game', 'tov_per_game', 'pf_per_game', 'pts_per_game']

advanced (33339, 30)
['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs', 'mp', 'per', 'ts_percent', 'x3p_ar', 'f_tr', 'orb_percent', 'drb_percent', 'trb_percent', 'ast_percent', 'stl_percent', 'blk_percent', 'tov_percent', 'usg_percent', 'ows', 'dws', 'ws', 'ws_48', 'obpm', 'dbpm', 'bpm', 'vorp']

team_stats (1907, 28)
['season', 'lg', 'team', 'abbreviation', 'playoffs', 'g', 'mp_per_game', 'fg_per_game', 'fga_per_game', 'fg_percent', 'x3p_per_game', 'x3pa_per_game', 'x3p_percent', 'x2p_per_game', 'x2pa_per_game', 'x2p

In [16]:
print(voting['type'].unique())

['all_defense' 'all_nba' 'all_rookie']


In [17]:
award_shares = pd.read_csv(f'{data_dir}/Player Award Shares.csv')
print(award_shares.shape)
print(award_shares.columns.tolist())

# find the column that identifies which award this row is for, and confirm MVP is in it
for col in award_shares.columns:
    if award_shares[col].dtype == object and award_shares[col].nunique() < 20:
        print(col, award_shares[col].unique())

team_summaries = pd.read_csv(f'{data_dir}/Team Summaries.csv')
print(team_summaries.shape)
print(team_summaries.columns.tolist())

(3465, 10)
['season', 'award', 'player', 'player_id', 'age', 'first', 'pts_won', 'pts_max', 'share', 'winner']
award ['nba clutch_poy' 'nba dpoy' 'nba mip' 'nba mvp' 'nba roy' 'nba smoy'
 'aba mvp' 'aba roy' 'baa roy']
(1907, 31)
['season', 'lg', 'team', 'abbreviation', 'playoffs', 'age', 'w', 'l', 'pw', 'pl', 'mov', 'sos', 'srs', 'o_rtg', 'd_rtg', 'n_rtg', 'pace', 'f_tr', 'x3p_ar', 'ts_percent', 'e_fg_percent', 'tov_percent', 'orb_percent', 'ft_fga', 'opp_e_fg_percent', 'opp_tov_percent', 'drb_percent', 'opp_ft_fga', 'arena', 'attend', 'attend_g']


In [21]:
import pandas as pd
import numpy as np

data_dir = '/content/drive/MyDrive/nba-mvp-predictor/data'

per_game = pd.read_csv(f'{data_dir}/Player Per Game.csv')
advanced = pd.read_csv(f'{data_dir}/Advanced.csv')
team_summaries = pd.read_csv(f'{data_dir}/Team Summaries.csv')
award_shares = pd.read_csv(f'{data_dir}/Player Award Shares.csv')

# Keep NBA only (drop old ABA/BAA rows — different era, different voting pool)
per_game = per_game[per_game['lg'] == 'NBA'].copy()
advanced = advanced[advanced['lg'] == 'NBA'].copy()
team_summaries = team_summaries[team_summaries['lg'] == 'NBA'].copy()

# Filter awards to MVP only
mvp_shares = award_shares[award_shares['award'] == 'nba mvp'].copy()
mvp_shares = mvp_shares[['season', 'player_id', 'pts_won', 'pts_max', 'share', 'winner']]
print("MVP rows:", mvp_shares.shape)

# Handle traded players: Basketball-Reference gives a combined "2TM"/"3TM" row
# plus individual team rows when a player is traded mid-season.
# Keep the combined stats row, but tag it with the team they played most games for.
def resolve_traded(df):
    def pick_rows(group):
        if len(group) == 1:
            return group
        multi = group[group['team'].astype(str).str.match(r'^\d+TM$')]
        singles = group[~group['team'].astype(str).str.match(r'^\d+TM$')]
        if len(multi) == 1 and len(singles) > 0:
            main_team = singles.loc[singles['g'].idxmax(), 'team']
            row = multi.copy()
            row['team'] = main_team
            return row
        return group.loc[[group['g'].idxmax()]]
    return df.groupby(['season', 'player_id'], group_keys=False).apply(pick_rows)

per_game_clean = resolve_traded(per_game)
advanced_clean = resolve_traded(advanced)
print("per_game:", per_game.shape, "->", per_game_clean.shape)
print("advanced:", advanced.shape, "->", advanced_clean.shape)

# Merge per_game + advanced (drop overlapping columns from advanced first)
overlap_cols = ['lg', 'player', 'age', 'team', 'pos', 'g', 'gs']
advanced_slim = advanced_clean.drop(columns=[c for c in overlap_cols if c in advanced_clean.columns])
players = per_game_clean.merge(advanced_slim, on=['season', 'player_id'], how='inner')
print("players merged:", players.shape)

# Merge in team win percentage
team_summaries['win_pct'] = team_summaries['w'] / (team_summaries['w'] + team_summaries['l'])
team_slim = team_summaries[['season', 'abbreviation', 'w', 'l', 'win_pct', 'srs', 'pace']]
players = players.merge(team_slim, left_on=['season', 'team'], right_on=['season', 'abbreviation'], how='left')
print("players + team:", players.shape, "| missing team match:", players['win_pct'].isna().sum())

# Merge in MVP share (0 for anyone who received no MVP votes that season)
players = players.merge(mvp_shares, on=['season', 'player_id'], how='left')
players['share'] = players['share'].fillna(0)
players['pts_won'] = players['pts_won'].fillna(0)
players['pts_max'] = players['pts_max'].fillna(0)
players['winner'] = players['winner'].fillna(False).infer_objects(copy=False)

print("final shape:", players.shape)
players.head()

MVP rows: (1057, 6)


/tmp/ipykernel_567/1820322385.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(['season', 'player_id'], group_keys=False).apply(pick_rows)


per_game: (31119, 32) -> (25319, 32)
advanced: (31119, 30) -> (25319, 30)
players merged: (25319, 53)
players + team: (25319, 59) | missing team match: 5
final shape: (25319, 63)


/tmp/ipykernel_567/1820322385.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(['season', 'player_id'], group_keys=False).apply(pick_rows)
/tmp/ipykernel_567/1820322385.py:60: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  players['winner'] = players['winner'].fillna(False).infer_objects(copy=False)


,season,lg,player,player_id,age,team,pos,g,gs,mp_per_game,...,abbreviation,w,l,win_pct,srs,pace,pts_won,pts_max,share,winner
0,1950,NBA,Curly Armstrong,armstcu01,31.0,FTW,NaN,63,NaN,NaN,...,FTW,40.0,28.0,0.588235,1.84,NaN,0.0,0.0,0.0,False
1,1950,NBA,Cliff Barker,barkecl01,29.0,INO,SG,49,NaN,NaN,...,INO,39.0,25.0,0.609375,2.59,NaN,0.0,0.0,0.0,False
2,1950,NBA,Leo Barnhorst,barnhle01,25.0,CHS,SF,67,NaN,NaN,...,CHS,40.0,28.0,0.588235,2.06,NaN,0.0,0.0,0.0,False
3,1950,NBA,Ed Bartels,barteed01,24.0,DNN,NaN,15,NaN,NaN,...,DNN,11.0,51.0,0.177419,-11.31,NaN,0.0,0.0,0.0,False
4,1950,NBA,Ralph Beard,beardra01,22.0,INO,NaN,60,NaN,NaN,...,INO,39.0,25.0,0.609375,2.59,NaN,0.0,0.0,0.0,False


In [19]:
import os
processed_dir = '/content/drive/MyDrive/nba-mvp-predictor/data/processed'
os.makedirs(processed_dir, exist_ok=True)
players.to_csv(f'{processed_dir}/player_season_mvp.csv', index=False)

In [22]:
missing = players[players['win_pct'].isna()]
print(missing[['season', 'player', 'team', 'share']])

     season       player team  share
741    1955  Rollen Hans  BLB    0.0
755    1955     Dan King  BLB    0.0
766    1955   Al McGuire  BLB    0.0
776    1955     Jim Neal  BLB    0.0
788    1955     Al Roges  BLB    0.0


In [23]:
players = players.dropna(subset=['win_pct']).reset_index(drop=True)
print(players.shape)

(25314, 63)


In [24]:
# Modern voting era only
model_df = players[players['season'] >= 1981].copy()

# Filter to legitimate candidates
model_df = model_df[(model_df['g'] >= 50) & (model_df['mp_per_game'] >= 20)].copy()

print(model_df.shape)
print(model_df['season'].min(), model_df['season'].max())
print("MVP winners in filtered set:", model_df['winner'].sum())
print("Total actual MVP winners (should equal number of seasons):", players[players['winner'] == True]['season'].nunique())

(8605, 63)
1981 2026
MVP winners in filtered set: 44
Total actual MVP winners (should equal number of seasons): 70


In [25]:
model_df = players[players['season'] >= 1981].copy()

# proxy for season length: max games played by any player that season
season_games = model_df.groupby('season')['g'].transform('max')

model_df = model_df[(model_df['g'] >= season_games * 0.6) & (model_df['mp_per_game'] >= 20)].copy()

print(model_df.shape)

# precise apples-to-apples check for exactly this range
expected = players[(players['season'] >= 1981) & (players['winner'] == True)]['season'].nunique()
actual = model_df['winner'].sum()
print(f"Expected MVP winners since 1981: {expected}")
print(f"Captured in filtered set: {actual}")

if actual < expected:
    missing_szn = set(players[(players['season']>=1981)&(players['winner']==True)]['season']) - set(model_df[model_df['winner']==True]['season'])
    print("Missing season(s):", missing_szn)

(8790, 63)
Expected MVP winners since 1981: 45
Captured in filtered set: 45


In [26]:
feature_cols = [
    'age', 'pts_per_game', 'ast_per_game', 'trb_per_game', 'stl_per_game',
    'blk_per_game', 'tov_per_game', 'fg_percent', 'x3p_percent', 'ft_percent',
    'per', 'ts_percent', 'usg_percent', 'ows', 'dws', 'ws', 'ws_48',
    'obpm', 'dbpm', 'bpm', 'vorp', 'win_pct', 'srs'
]

target_col = 'share'

corr = model_df[feature_cols + [target_col]].corr()[target_col].sort_values(ascending=False)
print(corr)

share           1.000000
vorp            0.501586
ws              0.449520
bpm             0.434447
ows             0.421863
per             0.420721
obpm            0.396088
ws_48           0.382267
pts_per_game    0.355668
dws             0.305410
usg_percent     0.292384
tov_per_game    0.269477
dbpm            0.214477
trb_per_game    0.203399
ast_per_game    0.200464
win_pct         0.189949
stl_per_game    0.186765
srs             0.179437
ts_percent      0.151178
blk_per_game    0.140941
fg_percent      0.122653
ft_percent      0.045825
age             0.020159
x3p_percent    -0.001021
Name: share, dtype: float64


In [27]:
train_df = model_df[model_df['season'] <= 2015].copy()
test_df = model_df[model_df['season'] > 2015].copy()

print("train seasons:", train_df['season'].min(), "-", train_df['season'].max(), "| rows:", train_df.shape[0])
print("test seasons:", test_df['season'].min(), "-", test_df['season'].max(), "| rows:", test_df.shape[0])
print("train MVP winners:", train_df['winner'].sum(), "| test MVP winners:", test_df['winner'].sum())

train seasons: 1981 - 2015 | rows: 6496
test seasons: 2016 - 2026 | rows: 2294
train MVP winners: 35 | test MVP winners: 10


In [28]:
train_df.to_csv(f'{processed_dir}/train.csv', index=False)
test_df.to_csv(f'{processed_dir}/test.csv', index=False)

In [29]:
print(train_df[feature_cols].isna().sum())

age               0
pts_per_game      0
ast_per_game      0
trb_per_game      0
stl_per_game      0
blk_per_game      0
tov_per_game      0
fg_percent        0
x3p_percent     411
ft_percent        0
per               0
ts_percent        0
usg_percent       0
ows               0
dws               0
ws                0
ws_48             0
obpm              0
dbpm              0
bpm               0
vorp              0
win_pct           0
srs               0
dtype: int64


In [30]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_train = train_df[feature_cols].fillna(0)
y_train = train_df[target_col]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df[target_col]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

pred_train = ridge.predict(X_train_scaled)
pred_test = ridge.predict(X_test_scaled)

print("Train RMSE:", np.sqrt(mean_squared_error(y_train, pred_train)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pred_test)))
print("Test R2:", r2_score(y_test, pred_test))

Train RMSE: 0.07138343757571618
Test RMSE: 0.07255734982349842
Test R2: 0.27027863662146856


In [31]:
test_eval = test_df.copy()
test_eval['pred_share'] = pred_test

hits = 0
seasons = sorted(test_eval['season'].unique())
for season in seasons:
    group = test_eval[test_eval['season'] == season]
    predicted_mvp = group.loc[group['pred_share'].idxmax(), 'player']
    actual_row = group[group['winner'] == True]
    actual_mvp = actual_row['player'].values[0] if len(actual_row) > 0 else None
    match = predicted_mvp == actual_mvp
    hits += match
    print(f"{season}: predicted={predicted_mvp!r}, actual={actual_mvp!r}, match={match}")

print(f"\nTop-1 hit rate: {hits}/{len(seasons)} = {hits/len(seasons):.1%}")

2016: predicted='Stephen Curry', actual='Stephen Curry', match=True
2017: predicted='Russell Westbrook', actual='Russell Westbrook', match=True
2018: predicted='LeBron James', actual='James Harden', match=False
2019: predicted='James Harden', actual='Giannis Antetokounmpo', match=False
2020: predicted='James Harden', actual='Giannis Antetokounmpo', match=False
2021: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
2022: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
2023: predicted='Nikola Jokić', actual='Joel Embiid', match=False
2024: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
2025: predicted='Nikola Jokić', actual='Shai Gilgeous-Alexander', match=False
2026: predicted='Nikola Jokić', actual=None, match=False

Top-1 hit rate: 5/11 = 45.5%


In [32]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

rf = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
pred_test_rf = rf.predict(X_test)

xgb = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
pred_test_xgb = xgb.predict(X_test)

In [33]:
def evaluate_predictions(df, preds, label):
    d = df.copy()
    d['pred_share'] = preds
    hits, valid_seasons = 0, 0
    for season in sorted(d['season'].unique()):
        group = d[d['season'] == season]
        if not group['winner'].any():
            continue
        valid_seasons += 1
        predicted_mvp = group.loc[group['pred_share'].idxmax(), 'player']
        actual_mvp = group.loc[group['winner'] == True, 'player'].values[0]
        match = predicted_mvp == actual_mvp
        hits += match
        print(f"[{label}] {season}: predicted={predicted_mvp!r}, actual={actual_mvp!r}, match={match}")
    rate = hits / valid_seasons
    print(f"[{label}] Top-1 hit rate: {hits}/{valid_seasons} = {rate:.1%}\n")
    return rate

ridge_rate = evaluate_predictions(test_df, pred_test, "Ridge")
rf_rate = evaluate_predictions(test_df, pred_test_rf, "RandomForest")
xgb_rate = evaluate_predictions(test_df, pred_test_xgb, "XGBoost")

[Ridge] 2016: predicted='Stephen Curry', actual='Stephen Curry', match=True
[Ridge] 2017: predicted='Russell Westbrook', actual='Russell Westbrook', match=True
[Ridge] 2018: predicted='LeBron James', actual='James Harden', match=False
[Ridge] 2019: predicted='James Harden', actual='Giannis Antetokounmpo', match=False
[Ridge] 2020: predicted='James Harden', actual='Giannis Antetokounmpo', match=False
[Ridge] 2021: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Ridge] 2022: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Ridge] 2023: predicted='Nikola Jokić', actual='Joel Embiid', match=False
[Ridge] 2024: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Ridge] 2025: predicted='Nikola Jokić', actual='Shai Gilgeous-Alexander', match=False
[Ridge] Top-1 hit rate: 5/10 = 50.0%

[RandomForest] 2016: predicted='Stephen Curry', actual='Stephen Curry', match=True
[RandomForest] 2017: predicted='James Harden', actual='Russell Westbrook', match=False
[Random

In [34]:
print("RF  Test RMSE:", np.sqrt(mean_squared_error(y_test, pred_test_rf)), "| R2:", r2_score(y_test, pred_test_rf))
print("XGB Test RMSE:", np.sqrt(mean_squared_error(y_test, pred_test_xgb)), "| R2:", r2_score(y_test, pred_test_xgb))

RF  Test RMSE: 0.050673531008888655 | R2: 0.6440765741669121
XGB Test RMSE: 0.05238357024304744 | R2: 0.6196491184311546


In [35]:
import pandas as pd

rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
xgb_importance = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Random Forest importance:\n", rf_importance.head(10))
print("\nXGBoost importance:\n", xgb_importance.head(10))

Random Forest importance:
 ws              0.329173
per             0.245122
win_pct         0.097171
vorp            0.066142
srs             0.035154
tov_per_game    0.024231
pts_per_game    0.023558
usg_percent     0.022655
ft_percent      0.018198
ws_48           0.017724
dtype: float64

XGBoost importance:
 per             0.343386
ws              0.178948
win_pct         0.073704
vorp            0.072648
obpm            0.059671
srs             0.036114
tov_per_game    0.033104
pts_per_game    0.021314
dws             0.020837
ast_per_game    0.020704
dtype: float32


In [36]:
eval_seasons = sorted(model_df[model_df['winner'] == True]['season'].unique())
print(f"Evaluating across {len(eval_seasons)} seasons")

def loso_cv(model_class, model_kwargs, label):
    hits = 0
    for season in eval_seasons:
        train = model_df[model_df['season'] != season]
        test = model_df[model_df['season'] == season]
        X_tr, y_tr = train[feature_cols].fillna(0), train[target_col]
        X_te = test[feature_cols].fillna(0)

        model = model_class(**model_kwargs)
        model.fit(X_tr, y_tr)
        test_eval = test.copy()
        test_eval['pred_share'] = model.predict(X_te)

        predicted_mvp = test_eval.loc[test_eval['pred_share'].idxmax(), 'player']
        actual_mvp = test_eval.loc[test_eval['winner'] == True, 'player'].values[0]
        hits += (predicted_mvp == actual_mvp)

    rate = hits / len(eval_seasons)
    print(f"[{label}] LOSO hit rate: {hits}/{len(eval_seasons)} = {rate:.1%}")
    return rate

rf_loso = loso_cv(RandomForestRegressor, {'n_estimators': 300, 'max_depth': 6, 'random_state': 42}, "RandomForest")
xgb_loso = loso_cv(XGBRegressor, {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'random_state': 42}, "XGBoost")

Evaluating across 45 seasons
[RandomForest] LOSO hit rate: 32/45 = 71.1%
[XGBoost] LOSO hit rate: 28/45 = 62.2%


In [37]:
final_train = model_df[model_df['season'] < 2026].copy()
X_final = final_train[feature_cols].fillna(0)
y_final = final_train[target_col]

final_model = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42)
final_model.fit(X_final, y_final)

print("Trained on", final_train['season'].nunique(), "seasons,", len(final_train), "player-rows")

Trained on 45 seasons, 8580 player-rows


In [38]:
import joblib, os

models_dir = '/content/drive/MyDrive/nba-mvp-predictor/models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(final_model, f'{models_dir}/rf_mvp_model.joblib')
joblib.dump(feature_cols, f'{models_dir}/feature_cols.joblib')

print(os.listdir(models_dir))

['rf_mvp_model.joblib', 'feature_cols.joblib']


In [39]:
app_data_path = f'{processed_dir}/mvp_model_data.csv'
model_df.to_csv(app_data_path, index=False)
print(model_df.shape)

(8790, 63)


In [40]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5, 0.7, 1.0],
}

groups = train_df['season']
gkf = GroupKFold(n_splits=5)

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=list(gkf.split(X_train, y_train, groups=groups)),
    random_state=42,
    n_jobs=-1,
)

rf_search.fit(X_train, y_train)
print("Best params:", rf_search.best_params_)
print("Best CV RMSE:", -rf_search.best_score_)

Best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 1.0, 'max_depth': 8}
Best CV RMSE: 0.04765425784461232


In [41]:
tuned_rf = rf_search.best_estimator_
pred_test_tuned = tuned_rf.predict(X_test)
evaluate_predictions(test_df, pred_test_tuned, "Tuned RF (holdout)")

tuned_params = {**rf_search.best_params_, 'random_state': 42}
tuned_loso = loso_cv(RandomForestRegressor, tuned_params, "Tuned RF (LOSO)")
print(f"Baseline LOSO: 71.1% (32/45) | Tuned LOSO: {tuned_loso:.1%}")

[Tuned RF (holdout)] 2016: predicted='Stephen Curry', actual='Stephen Curry', match=True
[Tuned RF (holdout)] 2017: predicted='Kawhi Leonard', actual='Russell Westbrook', match=False
[Tuned RF (holdout)] 2018: predicted='James Harden', actual='James Harden', match=True
[Tuned RF (holdout)] 2019: predicted='Giannis Antetokounmpo', actual='Giannis Antetokounmpo', match=True
[Tuned RF (holdout)] 2020: predicted='Giannis Antetokounmpo', actual='Giannis Antetokounmpo', match=True
[Tuned RF (holdout)] 2021: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Tuned RF (holdout)] 2022: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Tuned RF (holdout)] 2023: predicted='Nikola Jokić', actual='Joel Embiid', match=False
[Tuned RF (holdout)] 2024: predicted='Nikola Jokić', actual='Nikola Jokić', match=True
[Tuned RF (holdout)] 2025: predicted='Shai Gilgeous-Alexander', actual='Shai Gilgeous-Alexander', match=True
[Tuned RF (holdout)] Top-1 hit rate: 8/10 = 80.0%

[Tuned RF (L